<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/_Kids_High_School_Dot_Products.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Educational Guide: Visualizing Vector Projection

### Overview for Educators and Parents
This notebook contains a mathematical visualization created using the Manim animation engine. It is designed to help high school students grasp the geometric intuition behind the dot product, specifically focusing on the concept of vector projection.

### Concepts Explored
1. **Vector Projection**: Understanding how one vector casts a shadow onto another. This is a fundamental concept in linear algebra and physics (e.g., calculating work or resolving forces).
2. **The Dot Product**: Demonstrating that the dot product is not just an abstract calculation, but a measure of how much one vector points in the direction of another.
3. **Angular Relationships**: Visualizing how the magnitude and direction (sign) of a projection change as the angle between two vectors varies:
    * **Acute Angles**: A positive projection (shadow in the same direction).
    * **Right Angles (90 degrees)**: Zero projection (no shadow), demonstrating orthogonality.
    * **Obtuse Angles**: A negative projection (shadow in the opposite direction).

### Mathematical Foundation
The animation visualizes the scalar component of vector **a** along vector **b**, defined as:

comp_b(a) = |a| cos(theta) = (a . b) / |b|

### Target Audience
This resource is curated for High School students studying Pre-Calculus, Physics, or introductory Linear Algebra.

In [1]:
!apt-get update -qq
!apt-get install -y -qq libcairo2-dev libpango1.0-dev ffmpeg \
    texlive texlive-latex-extra texlive-fonts-extra dvisvgm > /dev/null
!pip install -q manim
print("Setup complete.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 37.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.1/652.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:

import os
# Re-importing after install to ensure fresh state
import numpy as np
from manim import *

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [2]:
%%manim -v WARNING ProjectionShadow
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
GitHub: github.com/zombimann/Mathematical-video-animations-and-visualization
Level: High School
Concept: Dot product as projection magnitude (a vector's shadow on another)

Story: A light shines straight down, perpendicular to vector b. The shadow that
vector a casts onto b has signed length  comp_b a = (a . b) / |b| = |a| cos(theta).
As a rotates, the shadow shrinks, vanishes at 90 degrees, then flips sign past it.
"""

from manim import *
import numpy as np

# ----------------------------------------------------------------------
# PARAMETRIC CONTROLS  (subject-matter names; tweak freely)
# ----------------------------------------------------------------------
# Format: YouTube Shorts, 9:16 portrait (consistent: 16 * 1080/1920 = 9.0)
config.pixel_width, config.pixel_height = 1080, 1920
config.frame_width, config.frame_height = 9.0, 16.0
config.frame_rate = 30

# Palette
SLATE   = "#1E222A"   # background
INK     = "#F0F0F0"   # light text
CYAN    = "#00F0FF"   # vector a / accents
ORANGE  = "#FFB347"   # positive shadow
MAGENTA = "#FF66CC"   # negative shadow
CARD_BG = "#11151B"   # panel fill
TEAL_BG = "#08343A"   # closing-card background (level primary, darkened)

# Geometry of the demonstration
ORIGIN_PT = np.array([-2.0, 0.6, 0.0])   # shared tail O of both vectors
B_HAT     = np.array([1.0, 0.0, 0.0])    # b points horizontally (the "ground")
B_LEN, A_LEN = 3.7, 2.7                   # vector lengths
THETA_START  = 38 * DEGREES
THETA_KEYS   = [80 * DEGREES, 90 * DEGREES, 130 * DEGREES, 38 * DEGREES]

# Pacing (seconds)
T_DRAW, T_BEAT, T_TILT = 0.8, 0.9, 1.8

config.background_color = SLATE


def fit_w(m, w):  # size by width  (predictable, crisp layout)
    m.scale_to_fit_width(w); return m

def fit_h(m, h):  # size by height
    m.scale_to_fit_height(h); return m


class ProjectionShadow(Scene):
    def construct(self):
        self.camera.background_color = SLATE
        theta = ValueTracker(THETA_START)

        # ---- live geometry -------------------------------------------
        def a_tip():
            ang = theta.get_value()
            return ORIGIN_PT + A_LEN * np.array([np.cos(ang), np.sin(ang), 0.0])

        def proj_scalar():                 # signed comp_b a = |a| cos(theta)
            return A_LEN * np.cos(theta.get_value())

        def foot():
            return ORIGIN_PT + proj_scalar() * B_HAT

        # ---- persistent chrome: watermark + level pill ---------------
        watermark = Text("© Mugambi Ndwiga / @craftsandengineering",
                         font_size=24, color=INK)
        fit_w(watermark, 5.5).set_opacity(0.6).to_corner(DR, buff=0.30)

        pill = RoundedRectangle(width=4.1, height=0.92, corner_radius=0.30,
                                stroke_width=0, fill_color=CYAN, fill_opacity=0.70)
        pill_txt = fit_w(Text("For Kids: High School", color=SLATE, weight=BOLD), 3.5)
        pill_grp = VGroup(pill, pill_txt.move_to(pill)).to_corner(UR, buff=0.30)
        self.add(watermark, pill_grp)

        # ---- title + hook --------------------------------------------
        title = fit_w(Text("Vector Projection", weight=BOLD, color=INK), 6.8)
        title.move_to([0, 6.0, 0])
        subtitle = fit_w(Text("the dot product as a shadow", color=CYAN), 6.2)
        subtitle.next_to(title, DOWN, buff=0.25)

        hook = fit_w(Text("Does a vector cast a shadow?", weight=BOLD, color=CYAN), 7.6)
        hook.move_to([0, 1.5, 0])
        self.play(Write(hook), run_time=1.1)
        self.wait(0.8)
        self.play(FadeOut(hook, shift=UP * 0.5), run_time=0.5)
        self.play(FadeIn(title, shift=DOWN * 0.3), run_time=0.7)
        self.play(FadeIn(subtitle), run_time=0.5)
        self.wait(0.3)

        # ---- caption strip (recycled per beat) -----------------------
        CAP_Y = 4.55
        def caption(s, color=INK, w=7.6):
            return fit_w(Text(s, color=color), w).move_to([0, CAP_Y, 0])

        # ---- reference vector b --------------------------------------
        b_vec = Arrow(ORIGIN_PT, ORIGIN_PT + B_LEN * B_HAT, buff=0, color=INK,
                      stroke_width=5, max_tip_length_to_length_ratio=0.12).set_z_index(3)
        b_lab = fit_h(MathTex(r"\vec b", color=INK), 0.55).next_to(b_vec.get_end(), RIGHT, buff=0.2)

        self.play(FadeOut(subtitle), run_time=0.3)
        cap = caption("Vector b sets the direction.")
        self.play(FadeIn(cap), GrowArrow(b_vec), FadeIn(b_lab), run_time=T_DRAW)
        self.wait(T_BEAT)

        # ---- leaning vector a + angle theta (live) -------------------
        a_vec = always_redraw(lambda: Arrow(ORIGIN_PT, a_tip(), buff=0, color=CYAN,
                              stroke_width=6, max_tip_length_to_length_ratio=0.14).set_z_index(5))
        a_lab = always_redraw(lambda: fit_h(MathTex(r"\vec a", color=CYAN), 0.55)
                              .next_to(a_tip(), UP, buff=0.15))
        arc = always_redraw(lambda: Angle(Line(ORIGIN_PT, ORIGIN_PT + B_HAT),
                            Line(ORIGIN_PT, a_tip()), radius=0.7, color=INK, stroke_width=3))
        th_lab = always_redraw(lambda: fit_h(MathTex(r"\theta", color=INK), 0.45)
                 .move_to(ORIGIN_PT + 1.15 * np.array([np.cos(theta.get_value() / 2),
                                                       np.sin(theta.get_value() / 2), 0.0])))

        self.play(FadeOut(cap), run_time=0.3)
        cap = caption("Vector a leans at angle theta.")
        self.play(FadeIn(cap), Create(a_vec), FadeIn(a_lab),
                  Create(arc), FadeIn(th_lab), run_time=T_DRAW)
        self.wait(T_BEAT)

        # ---- light ray straight down + perpendicular foot ------------
        ray = always_redraw(lambda: DashedLine(a_tip(), foot(), color=INK,
                            stroke_width=2.5, dash_length=0.13).set_z_index(4))
        dot = always_redraw(lambda: Dot(foot(), radius=0.07, color=INK).set_z_index(4))
        def right_angle():
            f, t = foot(), a_tip()
            if abs(t[1] - f[1]) < 0.18:
                return VGroup()
            try:
                return RightAngle(Line(f, f + 0.45 * RIGHT), Line(f, t),
                                  length=0.28, color=INK, stroke_width=2.5).set_z_index(4)
            except Exception:
                return VGroup()
        ra = always_redraw(right_angle)

        self.play(FadeOut(cap), run_time=0.3)
        cap = caption("Light shines straight down, perpendicular to b.")
        self.play(FadeIn(cap), Create(ray), FadeIn(dot), FadeIn(ra), run_time=T_DRAW)
        self.wait(T_BEAT)

        # ---- the shadow (glowing, sign-aware, layered BELOW b) -------
        shadow = always_redraw(lambda: Line(ORIGIN_PT, foot(), stroke_width=12,
                 color=(ORANGE if proj_scalar() >= 0 else MAGENTA)).set_z_index(2))
        self.play(FadeOut(cap), run_time=0.3)
        cap = caption("This shadow on b is the projection.", color=ORANGE)
        self.play(FadeIn(cap), Create(shadow), run_time=T_DRAW)
        self.wait(T_BEAT)

        # ---- formula card --------------------------------------------
        card = RoundedRectangle(width=8.6, height=3.4, corner_radius=0.30,
                                stroke_color=CYAN, stroke_width=2.5,
                                fill_color=CARD_BG, fill_opacity=0.92).move_to([0, -3.4, 0])
        eq = MathTex(r"\operatorname{comp}_{\vec b}\,\vec a", r"=",
                     r"\frac{\vec a \cdot \vec b}{\lvert \vec b\rvert}", r"=",
                     r"\lvert \vec a\rvert\cos\theta", color=INK)
        fit_w(eq, 7.6).move_to([0, -2.7, 0])

        read_lab = MathTex(r"\operatorname{comp}_{\vec b}\,\vec a \;=\;", color=INK)
        val = DecimalNumber(proj_scalar(), num_decimal_places=2,
                            include_sign=True, color=ORANGE)
        readout = fit_w(VGroup(read_lab, val).arrange(RIGHT, buff=0.2), 5.0).move_to([0, -4.35, 0])
        val.add_updater(lambda m: m.set_value(proj_scalar()).set_color(
            ORANGE if proj_scalar() >= 0 else MAGENTA))

        self.play(FadeOut(cap), FadeIn(card), run_time=0.5)
        self.play(Write(eq), run_time=1.2)
        self.play(Indicate(eq[2], color=ORANGE), Indicate(eq[4], color=CYAN), run_time=1.0)
        self.play(FadeIn(readout), run_time=0.5)
        self.wait(0.6)

        # ---- ACT: rotate a, watch the shadow respond -----------------
        cap = caption("Tilt a: the shadow shrinks.")
        self.play(FadeIn(cap), theta.animate.set_value(THETA_KEYS[0]),
                  run_time=T_TILT, rate_func=smooth)
        self.wait(0.4)

        self.play(FadeOut(cap), run_time=0.3)
        cap = caption("At 90 degrees the shadow vanishes:  a . b = 0.")
        self.play(FadeIn(cap), theta.animate.set_value(THETA_KEYS[1]),
                  run_time=T_TILT, rate_func=smooth)
        self.play(Flash(ORIGIN_PT, color=CYAN, line_length=0.4), run_time=0.8)
        self.wait(0.4)

        self.play(FadeOut(cap), run_time=0.3)
        cap = caption("Past 90 degrees it flips: negative.", color=MAGENTA)
        self.play(FadeIn(cap), theta.animate.set_value(THETA_KEYS[2]),
                  run_time=T_TILT, rate_func=smooth)
        self.wait(0.6)

        self.play(FadeOut(cap), theta.animate.set_value(THETA_KEYS[3]),
                  run_time=T_TILT, rate_func=smooth)

        # ---- recap ----------------------------------------------------
        recap = caption("The dot product measures how much a points along b.")
        self.play(FadeIn(recap), run_time=0.7)
        self.wait(1.2)

        val.clear_updaters()
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)

        # ---- closing card (<= 2 s) -----------------------------------
        close_bg = Rectangle(width=config.frame_width + 0.2, height=config.frame_height + 0.2,
                             fill_color=TEAL_BG, fill_opacity=1.0, stroke_width=0).set_z_index(10)
        made = fit_w(Text("Made by Mugambi Ndwiga", color=WHITE, weight=BOLD), 6.6)
        handle = fit_w(Text("@craftsandengineering", color=WHITE), 4.6)
        closing = VGroup(made, handle).arrange(DOWN, buff=0.5).move_to(ORIGIN).set_z_index(11)
        self.add(close_bg)
        self.play(FadeIn(closing), run_time=0.5)
        self.wait(1.0)
        self.play(FadeOut(closing), FadeOut(close_bg), run_time=0.3)

Manim Community v0.20.1